# Unified Colab Trainer (Fingerspelling + TSL-51)

This single notebook trains both tracks in one run:
1. Thai Fingerspelling (One-Stage-TFS)
2. TSL-51 word signs

Default data mode is **Drive + local copy** for faster training I/O.


In [ ]:
# ================================================================
# Unified training pipeline for both datasets
# ================================================================

import csv
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ----------------------------------------------------------------
# Paths and flags (edit here)
# ----------------------------------------------------------------
if IN_COLAB:
    REPO_ROOT = Path("/content/TSL")
    REPO_URL = "https://github.com/YOUR_USERNAME/TSL.git"  # set this to your repo
    DRIVE_ROOT = Path("/content/drive/MyDrive/TSL")
else:
    REPO_ROOT = Path.cwd().resolve()
    DRIVE_ROOT = REPO_ROOT / "_colab_drive_sim"

RUN_FINGERSPELLING = True
RUN_TSL51 = True

FORCE_REEXTRACT_FS = False
FORCE_LOCAL_SYNC_FS = True
FORCE_REFRESH_TSL51_CACHE = False

TSL51_MAX_SAMPLES = None  # set int to cap samples for quick experiments

FS_ZIP_DRIVE_PATH = DRIVE_ROOT / "datasets" / "one_stage_tfs.zip"
FS_EXTRACT_DRIVE_DIR = DRIVE_ROOT / "cache" / "one_stage_tfs_extracted"
FS_WORK_DIR = Path("/content/fs_dataset") if IN_COLAB else (REPO_ROOT / "_runtime" / "fs_dataset")
FS_FEATURE_CACHE = DRIVE_ROOT / "cache" / "keypoints_cache_v3.npz"
FS_ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "fingerspelling"

TSL51_REPO_ID = "Namonpas/thai-sign-language-tsl51"
TSL51_META_CACHE_DIR = DRIVE_ROOT / "cache" / "tsl51" / "metadata"
TSL51_FILE_CACHE_DIR = DRIVE_ROOT / "cache" / "tsl51" / "files"
TSL51_WORK_DIR = Path("/content/tsl51_cache") if IN_COLAB else (REPO_ROOT / "_runtime" / "tsl51_cache")
TSL51_FEATURE_CACHE = DRIVE_ROOT / "cache" / "tsl51_features_v3.npz"
TSL51_ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "tsl51"

for p in [
    DRIVE_ROOT,
    FS_ZIP_DRIVE_PATH.parent,
    FS_EXTRACT_DRIVE_DIR,
    FS_WORK_DIR,
    FS_FEATURE_CACHE.parent,
    FS_ARTIFACT_DIR,
    TSL51_META_CACHE_DIR,
    TSL51_FILE_CACHE_DIR,
    TSL51_WORK_DIR,
    TSL51_FEATURE_CACHE.parent,
    TSL51_ARTIFACT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if REPO_ROOT.exists():
        print("Repo exists, pulling latest:", REPO_ROOT)
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull"], check=False)
    else:
        if "YOUR_USERNAME" in REPO_URL:
            raise RuntimeError("Set REPO_URL before running in Colab.")
        print("Cloning repo:", REPO_URL)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("RUN_FINGERSPELLING:", RUN_FINGERSPELLING)
print("RUN_TSL51:", RUN_TSL51)

# ----------------------------------------------------------------
# Install deps only when missing
# ----------------------------------------------------------------
def ensure_pkg(import_name: str, pip_spec: str):
    import importlib.util
    if importlib.util.find_spec(import_name) is None:
        print("Installing", pip_spec)
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_spec])

ensure_pkg("numpy", "numpy>=1.24,<2.0")
ensure_pkg("cv2", "opencv-python>=4.8")
ensure_pkg("mediapipe", "mediapipe==0.10.14")
ensure_pkg("tensorflow", "tensorflow>=2.13,<2.17")
ensure_pkg("sklearn", "scikit-learn>=1.3")
ensure_pkg("joblib", "joblib>=1.3")
ensure_pkg("matplotlib", "matplotlib>=3.7")
ensure_pkg("tqdm", "tqdm>=4.66")
ensure_pkg("huggingface_hub", "huggingface_hub>=0.23")

import joblib
import numpy as np
import cv2
import mediapipe as mp
import tensorflow as tf

from tqdm import tqdm
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# ----------------------------------------------------------------
# Shared helper functions (feature contracts)
# ----------------------------------------------------------------
FEATURE_SIZE = 63
NUM_LANDMARKS = 21
FEATURE_DIM = 162
SEQ_LEN = 60

def extract_hand_landmarks(results):
    if not results.multi_hand_landmarks:
        return None
    if results.multi_handedness and len(results.multi_handedness) > 1:
        scores = [h.classification[0].score for h in results.multi_handedness]
        best_idx = int(np.argmax(scores))
    else:
        best_idx = 0
    hand_landmarks = results.multi_hand_landmarks[best_idx]
    coords = []
    for lm in hand_landmarks.landmark:
        coords.extend([lm.x, lm.y, lm.z])
    return np.array(coords, dtype=np.float32)

def normalize_landmarks(arr):
    arr = arr.reshape(NUM_LANDMARKS, 3)
    wrist = arr[0].copy()
    arr = arr - wrist
    hand_span = np.linalg.norm(arr[9])
    if hand_span < 1e-6:
        return None
    arr = arr / hand_span
    return arr.flatten().astype(np.float32)

def extract_and_normalize(results):
    raw = extract_hand_landmarks(results)
    if raw is None:
        return None
    return normalize_landmarks(raw)

CSV_POSE_LANDMARKS = (
    "l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist",
)
CSV_FACE_LANDMARKS = (
    "lbrow_outer", "lbrow_inner", "rbrow_inner", "rbrow_outer", "mouth_right", "mouth_left",
)

def tsl51_csv_column_names():
    cols = []
    for prefix in CSV_POSE_LANDMARKS + CSV_FACE_LANDMARKS:
        cols.extend((f"{prefix}_x", f"{prefix}_y", f"{prefix}_z"))
    for i in range(21):
        for hand in ("lh", "rh"):
            cols.extend((f"{hand}_x{i}", f"{hand}_y{i}", f"{hand}_z{i}"))
    if len(cols) != FEATURE_DIM:
        raise RuntimeError("Unexpected TSL-51 column count")
    return cols

def _parse_csv_float(value):
    if value is None:
        return 0.0
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return 0.0
    return float(text)

def _anchor_feature_frames(flat):
    out = np.empty_like(flat, dtype=np.float32)
    for t in range(flat.shape[0]):
        frame = flat[t].reshape(-1, 3)
        anchor = (frame[0] + frame[1]) / 2.0
        out[t] = (frame - anchor).reshape(FEATURE_DIM)
    return out

def read_landmark_csv(source):
    col_names = tsl51_csv_column_names()
    close_after = False
    if isinstance(source, bytes):
        handle = io.StringIO(source.decode("utf-8"))
    elif isinstance(source, (str, Path)):
        path = Path(source)
        text = str(source)
        if path.exists():
            handle = open(path, newline="", encoding="utf-8")
            close_after = True
        elif "\n" in text or text.lstrip().startswith("frame,"):
            handle = io.StringIO(text)
        else:
            raise FileNotFoundError(path)
    else:
        handle = io.StringIO(str(source))
    rows = []
    try:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            return np.zeros((0, FEATURE_DIM), dtype=np.float32)
        missing = [c for c in col_names if c not in reader.fieldnames]
        if missing:
            raise ValueError(f"CSV missing expected columns: {missing[:5]}")
        for row in reader:
            rows.append([_parse_csv_float(row.get(c)) for c in col_names])
    finally:
        if close_after:
            handle.close()
    if not rows:
        return np.zeros((0, FEATURE_DIM), dtype=np.float32)
    arr = np.asarray(rows, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0)
    return _anchor_feature_frames(arr)

def pad_truncate_sequence(features, seq_len=SEQ_LEN):
    if features.ndim != 2 or features.shape[1] != FEATURE_DIM:
        raise ValueError(f"Expected (T, {FEATURE_DIM}), got {features.shape}")
    t = features.shape[0]
    if t >= seq_len:
        return features[:seq_len].astype(np.float32, copy=False)
    pad = np.zeros((seq_len - t, FEATURE_DIM), dtype=np.float32)
    return np.concatenate([pad, features], axis=0).astype(np.float32)

def csv_to_sequence(source, seq_len=SEQ_LEN):
    return pad_truncate_sequence(read_landmark_csv(source), seq_len=seq_len)

def find_subdir_by_name(root: Path, dirname: str):
    for p in root.rglob(dirname):
        if p.is_dir():
            return p
    return None

def copy_tree(src: Path, dst: Path, force: bool = False):
    if force and dst.exists():
        shutil.rmtree(dst)
    if not dst.exists():
        shutil.copytree(src, dst)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

def save_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def load_metadata_rows(csv_path: Path):
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

# ----------------------------------------------------------------
# 1) Thai Fingerspelling track
# ----------------------------------------------------------------
if RUN_FINGERSPELLING:
    if not FS_ZIP_DRIVE_PATH.exists():
        raise FileNotFoundError(
            f"One-Stage-TFS zip not found: {FS_ZIP_DRIVE_PATH}\n"
            "Upload zip to this path on Drive, then rerun."
        )

    training_root_cached = find_subdir_by_name(FS_EXTRACT_DRIVE_DIR, "Training set")
    should_extract = FORCE_REEXTRACT_FS or training_root_cached is None
    if should_extract:
        if FS_EXTRACT_DRIVE_DIR.exists():
            shutil.rmtree(FS_EXTRACT_DRIVE_DIR)
        FS_EXTRACT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Extracting fingerspelling zip to Drive cache...")
        with zipfile.ZipFile(FS_ZIP_DRIVE_PATH, "r") as zf:
            zf.extractall(FS_EXTRACT_DRIVE_DIR)

    print("Syncing fingerspelling dataset to local runtime...")
    copy_tree(FS_EXTRACT_DRIVE_DIR, FS_WORK_DIR, force=FORCE_LOCAL_SYNC_FS)

    training_root = find_subdir_by_name(FS_WORK_DIR, "Training set")
    if training_root is None:
        raise RuntimeError("Cannot find 'Training set' after extraction/sync")
    test_root = training_root.parent / "Test set"
    if not test_root.exists():
        test_root = None

    def image_files(root_dir: Path):
        exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
        return [p for p in root_dir.rglob("*") if p.is_file() and p.suffix in exts]

    def extract_dataset(root_dir: Path, hands):
        X, y, class_names = [], [], []
        class_dirs = sorted([p for p in root_dir.iterdir() if p.is_dir()])
        for class_index, class_dir in enumerate(class_dirs):
            class_name = class_dir.name
            class_names.append(class_name)
            imgs = image_files(class_dir)
            for img_path in tqdm(imgs, desc=f"FS:{class_name}", leave=False):
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = hands.process(rgb)
                feat = extract_and_normalize(results)
                if feat is None:
                    continue
                X.append(feat)
                y.append(class_index)
        if not X:
            raise RuntimeError("No valid fingerspelling samples extracted")
        return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int32), class_names

    X_train_raw = y_train_raw = X_test_raw = y_test_raw = class_names = None
    use_cache = False
    if FS_FEATURE_CACHE.exists():
        try:
            data = np.load(FS_FEATURE_CACHE, allow_pickle=True)
            if int(data["feature_size"]) == FEATURE_SIZE:
                X_train_raw = data["X_train"]
                y_train_raw = data["y_train"]
                class_names = list(data["class_names"])
                if "X_test" in data:
                    X_test_raw = data["X_test"]
                if "y_test" in data:
                    y_test_raw = data["y_test"]
                use_cache = True
                print("Loaded fingerspelling features from cache:", FS_FEATURE_CACHE)
        except Exception as exc:
            print("Ignoring bad fingerspelling cache:", exc)

    if not use_cache:
        mp_hands = mp.solutions.hands
        with mp_hands.Hands(
            static_image_mode=True,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        ) as hands:
            X_train_raw, y_train_raw, class_names = extract_dataset(training_root, hands)
            if test_root is not None:
                X_test_raw, y_test_raw, _ = extract_dataset(test_root, hands)

        np.savez_compressed(
            FS_FEATURE_CACHE,
            feature_size=FEATURE_SIZE,
            X_train=X_train_raw,
            y_train=y_train_raw,
            class_names=np.asarray(class_names, dtype=object),
            X_test=np.asarray(X_test_raw if X_test_raw is not None else [], dtype=np.float32),
            y_test=np.asarray(y_test_raw if y_test_raw is not None else [], dtype=np.int32),
        )
        print("Saved fingerspelling cache:", FS_FEATURE_CACHE)

    if X_test_raw is not None and len(X_test_raw) > 0:
        X_tmp, X_val, y_tmp, y_val = train_test_split(
            X_train_raw, y_train_raw, test_size=0.12, stratify=y_train_raw, random_state=42
        )
        X_tr, y_tr = X_tmp, y_tmp
        X_test, y_test = X_test_raw, y_test_raw
    else:
        X_tmp, X_test, y_tmp, y_test = train_test_split(
            X_train_raw, y_train_raw, test_size=0.10, stratify=y_train_raw, random_state=42
        )
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tmp, y_tmp, test_size=0.111, stratify=y_tmp, random_state=42
        )

    def mirror_x(X):
        Xm = X.reshape(-1, 21, 3).copy()
        Xm[:, :, 0] *= -1.0
        return Xm.reshape(-1, 63)

    X_tr_aug = np.concatenate([X_tr, mirror_x(X_tr)], axis=0).astype(np.float32)
    y_tr_aug = np.concatenate([y_tr, y_tr], axis=0).astype(np.int32)

    scaler = StandardScaler()
    scaler.fit(X_tr_aug)
    X_tr_s = scaler.transform(X_tr_aug).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)

    num_classes = len(class_names)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(FEATURE_SIZE,)),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6),
    ]

    print("Training fingerspelling model...")
    history = model.fit(
        X_tr_s,
        y_tr_aug,
        validation_data=(X_val_s, y_val),
        epochs=120,
        batch_size=64,
        callbacks=callbacks,
        verbose=1,
    )

    test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
    y_pred = np.argmax(model.predict(X_test_s, verbose=0), axis=1)
    print(f"Fingerspelling test accuracy: {test_acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=class_names))

    FS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    fs_model_path = FS_ARTIFACT_DIR / "model.keras"
    fs_scaler_path = FS_ARTIFACT_DIR / "scaler.pkl"
    fs_labels_path = FS_ARTIFACT_DIR / "labels.json"
    fs_manifest_path = FS_ARTIFACT_DIR / "model_manifest.json"
    fs_tflite_path = FS_ARTIFACT_DIR / "model.tflite"

    model.save(fs_model_path)
    joblib.dump(scaler, fs_scaler_path)
    save_json(fs_labels_path, {str(i): name for i, name in enumerate(class_names)})

    fs_manifest = {
        "track": "thai_fingerspelling",
        "dataset": "One-Stage-TFS",
        "feature_size": FEATURE_SIZE,
        "num_classes": num_classes,
        "num_train": int(len(X_tr_s)),
        "num_val": int(len(X_val_s)),
        "num_test": int(len(X_test_s)),
        "test_accuracy": float(test_acc),
        "artifacts": ["model.keras", "model.tflite", "labels.json", "scaler.pkl"],
    }
    save_json(fs_manifest_path, fs_manifest)

    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_bytes = converter.convert()
        fs_tflite_path.write_bytes(tflite_bytes)
        print("Saved fingerspelling TFLite:", fs_tflite_path)
    except Exception as exc:
        print("Fingerspelling TFLite conversion skipped:", exc)

    print("Fingerspelling artifacts:", FS_ARTIFACT_DIR)

# ----------------------------------------------------------------
# 2) TSL-51 word-sign track
# ----------------------------------------------------------------
if RUN_TSL51:
    meta_files = [
        "metadata/expert_metadata.csv",
        "metadata/user_sign_metadata.csv",
    ]

    local_meta_paths = []
    for rel in meta_files:
        dst = TSL51_META_CACHE_DIR / Path(rel).name
        if FORCE_REFRESH_TSL51_CACHE or not dst.exists():
            print("Downloading metadata:", rel)
            local = hf_hub_download(TSL51_REPO_ID, rel, repo_type="dataset")
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local, dst)
        local_meta_paths.append(dst)

    rows = []
    for path in local_meta_paths:
        rows.extend(load_metadata_rows(path))

    rows = [r for r in rows if str(r.get("sign_id", "")).strip() and str(r.get("landmark_path", "")).strip()]
    rows_train = [r for r in rows if str(r["sign_id"]).strip() != "null_act"]
    if not rows_train:
        raise RuntimeError("No trainable TSL-51 rows after filtering null_act")

    if TSL51_MAX_SAMPLES is not None and TSL51_MAX_SAMPLES > 0:
        rows_train = rows_train[: int(TSL51_MAX_SAMPLES)]

    signature = f"{len(rows_train)}|{rows_train[0]['landmark_path']}|{rows_train[-1]['landmark_path']}"
    use_tsl_cache = False
    X_all = y_labels = class_names = None
    if TSL51_FEATURE_CACHE.exists() and not FORCE_REFRESH_TSL51_CACHE:
        try:
            data = np.load(TSL51_FEATURE_CACHE, allow_pickle=True)
            if str(data["signature"]) == signature and int(data["feature_dim"]) == FEATURE_DIM and int(data["seq_len"]) == SEQ_LEN:
                X_all = data["X"]
                y_labels = data["y"]
                class_names = list(data["class_names"])
                use_tsl_cache = True
                print("Loaded TSL-51 features from cache:", TSL51_FEATURE_CACHE)
        except Exception as exc:
            print("Ignoring bad TSL-51 cache:", exc)

    def ensure_landmark_local(rel_path: str):
        rel = str(rel_path).replace("\\", "/").strip()
        drive_file = TSL51_FILE_CACHE_DIR / rel
        if FORCE_REFRESH_TSL51_CACHE and drive_file.exists():
            drive_file.unlink()
        if not drive_file.exists():
            remote_local = hf_hub_download(TSL51_REPO_ID, rel, repo_type="dataset")
            drive_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(remote_local, drive_file)
        local_file = TSL51_WORK_DIR / rel
        if not local_file.exists():
            local_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(drive_file, local_file)
        return local_file

    if not use_tsl_cache:
        class_names = sorted({str(r["sign_id"]).strip() for r in rows_train})
        class_to_idx = {name: i for i, name in enumerate(class_names)}
        X_list, y_list = [], []
        bad_rows = 0
        for row in tqdm(rows_train, desc="TSL51 sequences"):
            try:
                lp = ensure_landmark_local(row["landmark_path"])
                seq = csv_to_sequence(lp, seq_len=SEQ_LEN)
                X_list.append(seq)
                y_list.append(class_to_idx[str(row["sign_id"]).strip()])
            except Exception:
                bad_rows += 1
        if not X_list:
            raise RuntimeError("No valid TSL-51 sequences were loaded")
        if bad_rows > 0:
            print("Skipped bad TSL-51 rows:", bad_rows)

        X_all = np.asarray(X_list, dtype=np.float32)
        y_labels = np.asarray(y_list, dtype=np.int32)

        np.savez_compressed(
            TSL51_FEATURE_CACHE,
            signature=signature,
            feature_dim=FEATURE_DIM,
            seq_len=SEQ_LEN,
            X=X_all,
            y=y_labels,
            class_names=np.asarray(class_names, dtype=object),
        )
        print("Saved TSL-51 feature cache:", TSL51_FEATURE_CACHE)

    X_tmp, X_test, y_tmp, y_test = train_test_split(
        X_all, y_labels, test_size=0.10, stratify=y_labels, random_state=42
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tmp, y_tmp, test_size=0.111, stratify=y_tmp, random_state=42
    )

    scaler_seq = StandardScaler()
    scaler_seq.fit(X_tr.reshape(-1, FEATURE_DIM))

    def scale_sequences(X):
        return scaler_seq.transform(X.reshape(-1, FEATURE_DIM)).reshape(-1, SEQ_LEN, FEATURE_DIM).astype(np.float32)

    X_tr_s = scale_sequences(X_tr)
    X_val_s = scale_sequences(X_val)
    X_test_s = scale_sequences(X_test)

    num_classes = len(class_names)
    seq_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(SEQ_LEN, FEATURE_DIM)),
        tf.keras.layers.Masking(mask_value=0.0),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(96, return_sequences=True)),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])
    seq_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6),
    ]

    print("Training TSL-51 model...")
    seq_model.fit(
        X_tr_s,
        y_tr,
        validation_data=(X_val_s, y_val),
        epochs=120,
        batch_size=32,
        callbacks=callbacks,
        verbose=1,
    )

    test_loss, test_acc = seq_model.evaluate(X_test_s, y_test, verbose=0)
    y_pred = np.argmax(seq_model.predict(X_test_s, verbose=0), axis=1)
    print(f"TSL-51 test accuracy: {test_acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=class_names))

    TSL51_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    tsl_model_path = TSL51_ARTIFACT_DIR / "tsl51_model.keras"
    tsl_scaler_path = TSL51_ARTIFACT_DIR / "tsl51_scaler.pkl"
    tsl_labels_path = TSL51_ARTIFACT_DIR / "tsl51_labels.json"
    tsl_manifest_path = TSL51_ARTIFACT_DIR / "tsl51_model_manifest.json"
    tsl_tflite_path = TSL51_ARTIFACT_DIR / "tsl51_model.tflite"

    seq_model.save(tsl_model_path)
    joblib.dump(scaler_seq, tsl_scaler_path)
    save_json(tsl_labels_path, {str(i): name for i, name in enumerate(class_names)})

    tsl_manifest = {
        "track": "tsl51_word_signs",
        "dataset": TSL51_REPO_ID,
        "feature_dim": FEATURE_DIM,
        "seq_len": SEQ_LEN,
        "num_classes": num_classes,
        "num_train": int(len(X_tr_s)),
        "num_val": int(len(X_val_s)),
        "num_test": int(len(X_test_s)),
        "test_accuracy": float(test_acc),
        "artifacts": ["tsl51_model.keras", "tsl51_model.tflite", "tsl51_labels.json", "tsl51_scaler.pkl"],
    }
    save_json(tsl_manifest_path, tsl_manifest)

    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(seq_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_bytes = converter.convert()
        tsl_tflite_path.write_bytes(tflite_bytes)
        print("Saved TSL-51 TFLite:", tsl_tflite_path)
    except Exception as exc:
        print("TSL-51 TFLite conversion skipped:", exc)

    print("TSL-51 artifacts:", TSL51_ARTIFACT_DIR)

print("All selected tracks completed.")
